# Exercise: Build a Flights Agent with DSPy

In this exercise, you will build an AI agent with [DSPy](https://dspy.ai/). AI agents are systems that can autonomously perceive their environment, make decisions, and take actions to achieve specific goals.

Unlike a single model prompt, an agent typically follows a loop of reasoning, planning, and acting — often integrating tools like search engines, APIs, or memory to complete complex tasks.

This exercise focuses on **ReAct** (**Re**asoning and **Act**ing):

- The LM receives a task description and a list of tools.
- At each step, it decides whether to call a tool for more observations or to produce the final output.

You will build a simple airline customer service agent that can:

- Book new trips on behalf of the user.
- Modify existing trips, including flight change and cancellation.
- Raise a customer support ticket for tasks it cannot handle.

Run the scaffold cells first, then complete each `___` blank in the exercise cells below.


## Setup

### Install dependencies

Before starting, install the required packages:

```bash
!pip install -qU dspy pydantic
```

### MLflow DSPy Integration

Set up MLflow Tracing to understand what's happening under the hood.

<a href="https://mlflow.org/">MLflow</a> is an LLMOps tool that natively integrates with DSPy and offers explainability and experiment tracking. You can use MLflow to visualize prompts and optimization progress as traces to understand DSPy's behavior better.

![MLflow Trace](../assets/mlflow-tracing-customer-service-agent.png)

1. Install MLflow

```bash
%pip install mlflow>=3.0.0
```

2. Start MLflow UI in a separate terminal

```bash
mlflow ui --port 5000 --backend-store-uri sqlite:///mlruns.db
```

3. Connect the notebook to MLflow

```python
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("DSPy")
```

4. Enable tracing.

```python
mlflow.dspy.autolog()
```

To learn more, visit [MLflow DSPy Documentation](https://mlflow.org/docs/latest/llms/dspy/index.html).

## Define Tools

We need a list of tools so the agent can behave like a human airline service agent:

- `fetch_flight_info`: get flight information for certain dates.
- `pick_flight`: pick the best flight based on some criteria.
- `book_flight`: book a flight on behalf of the user.
- `fetch_itinerary`: get the information of a booked itinerary.
- `cancel_itinerary`: cancel a booked itinerary.
- `get_user_info`: get users' information.
- `file_ticket`: file a backlog ticket to have a human assist.


### Define data structure

Before defining the tools, we need a data structure. In production, this would be the database schema. For this demo, we use [Pydantic models](https://docs.pydantic.dev/latest/concepts/models/) for simplicity.

In [ ]:
from pydantic import BaseModel  # https://docs.pydantic.dev/latest/


class Date(BaseModel):
    # LLMs struggle to emit valid `datetime.datetime` values, so we use a simple custom type.
    year: int
    month: int
    day: int
    hour: int


class UserProfile(BaseModel):
    user_id: str
    name: str
    email: str


class Flight(BaseModel):
    flight_id: str
    date_time: Date
    origin: str
    destination: str
    duration: float
    price: float


class Itinerary(BaseModel):
    confirmation_number: str
    user_profile: UserProfile
    flight: Flight


class Ticket(BaseModel):
    user_request: str
    user_profile: UserProfile

### Create dummy data

Create a few flights and users, and initialize empty dictionaries for itineraries and customer support tickets.

In [ ]:
user_database = {
    "Adam": UserProfile(user_id="1", name="Adam", email="adam@gmail.com"),
    "Bob": UserProfile(user_id="2", name="Bob", email="bob@gmail.com"),
    "Chelsie": UserProfile(user_id="3", name="Chelsie", email="chelsie@gmail.com"),
    "David": UserProfile(user_id="4", name="David", email="david@gmail.com"),
}

flight_database = {
    "DA123": Flight(
        flight_id="DA123",  # DSPy Airline 123
        origin="SFO",
        destination="JFK",
        date_time=Date(year=2025, month=9, day=1, hour=1),
        duration=3,
        price=200,
    ),
    "DA125": Flight(
        flight_id="DA125",
        origin="SFO",
        destination="JFK",
        date_time=Date(year=2025, month=9, day=1, hour=7),
        duration=9,
        price=500,
    ),
    "DA456": Flight(
        flight_id="DA456",
        origin="SFO",
        destination="SNA",
        date_time=Date(year=2025, month=10, day=1, hour=1),
        duration=2,
        price=100,
    ),
    "DA460": Flight(
        flight_id="DA460",
        origin="SFO",
        destination="SNA",
        date_time=Date(year=2025, month=10, day=1, hour=9),
        duration=2,
        price=120,
    ),
}

# In-memory stores the agent will read from and write to.
itinerary_database = {}
ticket_database = {}

### Define the tools

For `dspy.ReAct` to work, every tool function should:

- Have a docstring that describes what the tool does (unless the name is self-explanatory).
- Have type hints on arguments so the LM can generate correctly formatted tool calls.

In [ ]:
import random
import string


def fetch_flight_info(date: Date, origin: str, destination: str):
    """Fetch flight information from origin to destination on the given date"""
    flights = []

    for flight_id, flight in flight_database.items():
        if (
            flight.date_time.year == date.year
            and flight.date_time.month == date.month
            and flight.date_time.day == date.day
            and flight.origin == origin
            and flight.destination == destination
        ):
            flights.append(flight)
    if len(flights) == 0:
        raise ValueError("No matching flight found!")
    return flights


def fetch_itinerary(confirmation_number: str):
    """Fetch a booked itinerary information from database"""
    return itinerary_database.get(confirmation_number)


def pick_flight(flights: list[Flight]):
    """Pick up the best flight that matches users' request. we pick the shortest, and cheaper one on ties."""
    sorted_flights = sorted(
        flights,
        key=lambda x: (
            x.get("duration") if isinstance(x, dict) else x.duration,
            x.get("price") if isinstance(x, dict) else x.price,
        ),
    )
    return sorted_flights[0]


def _generate_id(length=8):
    chars = string.ascii_lowercase + string.digits
    return "".join(random.choices(chars, k=length))


def book_flight(flight: Flight, user_profile: UserProfile):
    """Book a flight on behalf of the user."""
    confirmation_number = _generate_id()
    while confirmation_number in itinerary_database:
        confirmation_number = _generate_id()
    itinerary_database[confirmation_number] = Itinerary(
        confirmation_number=confirmation_number,
        user_profile=user_profile,
        flight=flight,
    )
    return confirmation_number, itinerary_database[confirmation_number]


def cancel_itinerary(confirmation_number: str, user_profile: UserProfile):
    """Cancel an itinerary on behalf of the user."""
    if confirmation_number in itinerary_database:
        del itinerary_database[confirmation_number]
        return
    raise ValueError("Cannot find the itinerary, please check your confirmation number.")


def get_user_info(name: str):
    """Fetch the user profile from database with given name."""
    return user_database.get(name)


def file_ticket(user_request: str, user_profile: UserProfile):
    """File a customer support ticket if this is something the agent cannot handle."""
    ticket_id = _generate_id(length=6)
    ticket_database[ticket_id] = Ticket(
        user_request=user_request,
        user_profile=user_profile,
    )
    return ticket_id

## Create ReAct Agent

Create the ReAct agent with `dspy.ReAct`. You provide:

- A **signature** that defines the task, inputs, and outputs.
- A **tools** list the agent can call during its reasoning loop.

## Exercise 1: Define the agent signature

Define a DSPy signature class for the airline customer service agent.

- Keep the docstring as written — it tells the LM its role.
- Add an input field named `user_request`.
- Add an output field named `process_result` with the given description.

In [ ]:
import dspy  # https://dspy.ai/


class DSPyAirlineCustomerService(dspy.Signature):
    """You are an airline customer service agent that helps user book and manage flights."""

    user_request: str = dspy.___()
    process_result: str = dspy.___(
        desc=(
            "Message that summarizes the process result, and the information users need, e.g., the "
            "confirmation_number if a new flight is booked."
        )
    )

## Exercise 2: Wire up `dspy.ReAct`

Create a ReAct agent that uses your signature and the tools defined above.

- Pass `DSPyAirlineCustomerService` as the signature.
- Include all seven tool functions in the `tools` list.

In [ ]:
agent = dspy.___(
    ___,  # your Signature class
    tools=[
        ___,
        ___,
        ___,
        ___,
        ___,
        ___,
        ___,
    ],
)

## Use the Agent

To interact with the agent, pass the user's request through the `user_request` input field defined in your signature.

## Exercise 3: Configure the language model

Select a language model and set up your API key. We use `gpt-4o-mini` here, but you can change to other models.

For configuration options, see the [DSPy language model guide](https://dspy.ai/learn/programming/language_models/).


In [ ]:
dspy.configure(lm=dspy.LM("___"))

### Book a flight

Run the cell below to book a flight for Adam. Note the `confirmation_number` in the output — you will need it in Exercise 5.

In [ ]:
result = agent(
    user_request="please help me book a flight from SFO to JFK on 09/01/2025, my name is Adam",
)
print(result)

Verify the booked itinerary was stored in the database.

In [ ]:
print(itinerary_database)

### Interpret the result

The returned `Prediction` contains:

- `process_result`: the user-facing message defined in your signature.
- `reasoning`: a summary of why the agent took the actions it did.
- `trajectory`: a step-by-step record of the ReAct loop, including:
  - Reasoning (thought) at each step.
  - Tools picked by the LM at each step.
  - Arguments for each tool call.
  - Tool execution results at each step.

Behind the scenes, `dspy.ReAct` runs a loop that accumulates tool-call information along with the task description and sends it to the LM until it hits `max_iters` or the LM calls `finish`.

## Exercise 4: Inspect the ReAct loop

Use `dspy.inspect_history()` to see the LM prompts and responses for each ReAct step from the booking above.

- Pass `n=10` to show the last 10 LM calls.

In [ ]:
dspy.___(n=___)

In each LM call, the user message includes the task description and the trajectory of previous tool calls.

## Exercise 5: Modify an existing itinerary

Ask the agent to change Adam's booked flight to DA125 on 09/01.

- Set `confirmation_number` to the value from the booking result above.
- Run the agent with the provided request string.

In [ ]:
confirmation_number = "___"  # copy from the booking result above

result = agent(
    user_request=(
        f"i want to take DA125 instead on 09/01, please help me modify my itinerary {confirmation_number}"
    ),
)
print(result)

## Conclusion

Congrats on finishing the exercise! You built a customer service agent with DSPy. The key ideas are:

- Define tools as Python functions with **docstrings and type hints**.
- Provide the tools to `dspy.ReAct` along with a signature that defines the task.
- Invoke the agent with the input fields from your signature — it will run the reasoning-and-acting loop behind the scenes.

## References

- [Build AI Agents with DSPy](https://dspy.ai/tutorials/customer_service_agent/)